In [11]:
import sys
sys.path.append("..")

In [12]:
import tqdm
import torch
import pickle
import warnings
import numpy as np
import pandas as pd
import torch.optim as optim
from copy import deepcopy
import plotly.express as px
import plotly.graph_objects as go

from src.data import *
from src.utils import *
from src.model import *
from src.recourse import *

warnings.filterwarnings('ignore')

In [13]:
def runSparsity (params: dict, diff_const = 1e-2):
    first_time = True

    for algorithm in params['algorithms']:
        for v_alpha in params['alphas']:
            for v_lamb in params['lambdas']:
                for seed in params['seeds']:

                    file_path = f"../results/recourse/{params['base_model']}_{params['data']}_{algorithm}_{v_lamb}_{v_alpha}_{seed}.pkl"
                    data_tmp = pd.read_pickle(file_path)
                    if params['include_mask'] and "L1PSD" in params["algorithms"] and algorithm != "L1PSD":
                        data_l1psd = pd.read_pickle(f"../results/recourse/{params['base_model']}_{params['data']}_L1PSD_0.1_0.1_{seed}.pkl")
                        mask = data_l1psd["i"].to_numpy()
                        data_tmp = data_tmp.iloc[mask]
                    else:
                        data_l1psd = pd.read_pickle(f"../results/recourse/{params['base_model']}_{params['data']}_L1PSD_0.1_0.1_{seed}.pkl")
                        mask_i = data_l1psd["i"].to_numpy()
                        if params['data'] == "sba" and params['base_model'] == "lr" and v_alpha == 0.5:
                            data_tmp = data_tmp[data_tmp['i'].isin(mask_i)]    
                        elif params['data'] == "sba" and params['base_model'] == "nn" and v_alpha == 0.5:
                            data_tmp = data_tmp[data_tmp['i'].isin(mask_i)]

                    if first_time:
                        data = data_tmp.copy(deep=True)
                        first_time = False
                    else:
                        data = pd.concat((data, data_tmp), ignore_index=True)

                    print(f"Read {file_path}")

    data['diff'] = np.abs(data['x_r'] - data['x_0'])
    data['diff_count'] = data['diff'].apply(lambda x: np.sum(x > diff_const))
    return data

In [ ]:
params = {}
# 'lr', 'nn'
params['base_model'] = 'nn'
# 'synthetic', 'german', 'sba'
params['data'] = 'sba'
params['seeds'] = range(5)
# 'Alg1', 'L1PSD', 'ROARLInf', 'ROARL1'
params['algorithms'] = ['Alg1', 'L1PSD', 'ROARLInf', 'ROARL1']
params['include_mask'] = True
# 'add', 'multi'
params['s_method'] = "add"


params['alphas']= [0.1]
# german_lr (alpha=0.1)
# params['lambdas'] = [0.5,0.3,0.1,0.04,0.01,0.004,0.001]
# german_nn (alpha=0.1)
# params['lambdas'] = [0.7, 0.3, 0.1, 0.05, 0.01, 0.001]
# sba_lr (alpha=0.1)
# params['lambdas'] = [0.001, 0.01, 0.1, 0.7, 1.4, 2.1, 2.8, 3.5]
# sba_nn (alpha=0.1)
params['lambdas'] = [0.001, 0.01, 0.1, 0.7, 1.4, 2.1, 2.8, 3.5]

# params['alphas']= [0.5]
# # german_lr (alpha=0.5)
# # params['lambdas'] = [0.3,0.04, 0.004, 0.1, 0.01]
# # german_nn (alpha=0.5)
# # params['lambdas'] = [3.0, 0.7, 0.3, 0.1, 0.05, 0.01]
# # sba_lr (alpha=0.5)
# # params['lambdas'] = [0.001, 0.01, 0.1, 0.7, 1.4, 2.1, 2.8, 3.5]
# # sba_nn (alpha=0.5)
# # params['lambdas'] = [0.01, 0.1, 0.7, 2.1, 3.5]

sparsity_lambda = deepcopy(params['lambdas'])
sparsity_lambda.sort(reverse=True)
diff_const = 1e-2
df_results = runSparsity(params, diff_const=diff_const)

Read ../results/recourse/nn_sba_Alg1_0.001_0.1_0.pkl
Read ../results/recourse/nn_sba_Alg1_0.001_0.1_1.pkl
Read ../results/recourse/nn_sba_Alg1_0.001_0.1_2.pkl
Read ../results/recourse/nn_sba_Alg1_0.001_0.1_3.pkl
Read ../results/recourse/nn_sba_Alg1_0.001_0.1_4.pkl
Read ../results/recourse/nn_sba_Alg1_0.01_0.1_0.pkl
Read ../results/recourse/nn_sba_Alg1_0.01_0.1_1.pkl
Read ../results/recourse/nn_sba_Alg1_0.01_0.1_2.pkl
Read ../results/recourse/nn_sba_Alg1_0.01_0.1_3.pkl
Read ../results/recourse/nn_sba_Alg1_0.01_0.1_4.pkl
Read ../results/recourse/nn_sba_Alg1_0.1_0.1_0.pkl
Read ../results/recourse/nn_sba_Alg1_0.1_0.1_1.pkl
Read ../results/recourse/nn_sba_Alg1_0.1_0.1_2.pkl
Read ../results/recourse/nn_sba_Alg1_0.1_0.1_3.pkl
Read ../results/recourse/nn_sba_Alg1_0.1_0.1_4.pkl
Read ../results/recourse/nn_sba_Alg1_0.7_0.1_0.pkl
Read ../results/recourse/nn_sba_Alg1_0.7_0.1_1.pkl
Read ../results/recourse/nn_sba_Alg1_0.7_0.1_2.pkl
Read ../results/recourse/nn_sba_Alg1_0.7_0.1_3.pkl
Read ../results/

In [86]:
df_results['percent_diff'] = np.abs(df_results['x_r'] - df_results['x_0']) / (np.abs(df_results['x_0']) + 1e-7)
df_results['percent_diff'].apply(lambda x: np.sum(x > 0.01))


0      1
1      1
2      1
3      1
4      1
      ..
955    3
956    2
957    4
958    4
959    0
Name: percent_diff, Length: 960, dtype: int64

In [45]:
df_results['algorithm'] = df_results['algorithm'].replace('alg1',"Alg1")
df_results['algorithm'] = df_results['algorithm'].replace('ROAR',"ROARLInf")

df_results_mean = df_results.groupby(['algorithm', 'alpha', 'lambda'], as_index=False).mean()

In [46]:
# df_results_mean = df_results_mean.sort_values(['alpha', 'lambda', 'algorithm'])
# df_results_mean_sam = df_results_mean.iloc[0:len(params['algorithms'])]
# if "ROARLInf" in params['algorithms'] and "ROARL1" in params['algorithms']:
#     df_results_mean_sam.iloc[[2,3]] = df_results_mean_sam.iloc[[3,2]]
# stacked  = np.stack(df_results_mean_sam['diff'].apply(lambda x: np.where(x > diff_const, x, 0)))

# fig = px.imshow(stacked.round(1), 
#             color_continuous_scale="Reds",
#             y=df_results_mean_sam['algorithm'].to_list(),
#             text_auto=True,
#             labels=dict(x='Features', y='Algorithms', color='Avg Cost'),
#             title=f"German LR Alpha={df_results_mean_sam['alpha'].unique()} Lambda={df_results_mean_sam['lambda'].unique()}")

# fig.show()

In [68]:
df_results_mean_histo = df_results_mean.copy(deep=True)
df_results_mean_histo['lambda_str'] = df_results_mean_histo['lambda'].astype(str)

df_results_mean = df_results_mean.sort_values(['alpha', 'lambda', 'algorithm'], ascending=[True, True, False])
df_results_mean

,algorithm,alpha,lambda,seed,i,x_0,x_r,theta_0,diff,diff_count
24,ROARLInf,0.1,0.001,2.0,19.133333,"[-0.16719666666666663, -0.32093666666666676, 0...","[0.07299999396006267, -0.864340082804362, 0.01...","[0.08763999999999997, -0.40253666666666654, -0...","[0.38540999774102397, 0.5434033390918525, 0.53...",23.000000
16,ROARL1,0.1,0.001,2.0,19.133333,"[-0.16719666666666663, -0.32093666666666676, 0...","[-0.022826663653055825, -0.7419399897257487, 0...","[0.08763999999999997, -0.40253666666666654, -0...","[0.40100333057185, 0.42100333558956765, 0.4215...",23.966667
8,L1PSD,0.1,0.001,2.0,19.133333,"[-0.16719666666666663, -0.32093666666666676, 0...","[-0.16719333333333328, -0.3209400000000001, 0....","[0.09798, -0.4140266666666666, -0.51021, 0.128...","[3.3333333333334286e-06, 3.333333333332966e-06...",1.500000
0,Alg1,0.1,0.001,2.0,19.133333,"[-0.16719666666666663, -0.32093666666666676, 0...","[-0.1655633333333333, -0.35427333333333344, 0....","[0.08763999999999997, -0.40253666666666654, -0...","[0.0016333333333333334, 0.03333666666666667, 0...",1.666667
25,ROARLInf,0.1,0.010,2.0,19.133333,"[-0.16719666666666663, -0.32093666666666676, 0...","[0.042466668287913005, -0.8409666061401367, 0....","[0.08763999999999997, -0.40253666666666654, -0...","[0.3063633276096676, 0.5200300022271491, 0.522...",21.400000
17,ROARL1,0.1,0.010,2.0,19.133333,"[-0.16719666666666663, -0.32093666666666676, 0...","[-0.023903322219848634, -0.7225366592407226, 0...","[0.08763999999999997, -0.40253666666666654, -0...","[0.25203999231641494, 0.40159999417702347, 0.4...",22.033333
9,L1PSD,0.1,0.010,2.0,19.133333,"[-0.16719666666666663, -0.32093666666666676, 0...","[-0.16719666666666663, -0.3209400000000001, 0....","[0.09798, -0.4140266666666666, -0.51021, 0.128...","[0.0, 3.333333333332966e-06, 0.0, 0.0, 3.33333...",1.133333
1,Alg1,0.1,0.010,2.0,19.133333,"[-0.16719666666666663, -0.32093666666666676, 0...","[-0.1655633333333333, -0.35427333333333344, 0....","[0.08763999999999997, -0.40253666666666654, -0...","[0.0016333333333333334, 0.03333666666666667, 0...",1.666667
26,ROARLInf,0.1,0.100,2.0,19.133333,"[-0.16719666666666663, -0.32093666666666676, 0...","[-0.1671233336130778, -0.42275660832722983, 0....","[0.08763999999999997, -0.40253666666666654, -0...","[0.00030665852859616816, 0.10219333946704863, ...",10.933333
18,ROARL1,0.1,0.100,2.0,19.133333,"[-0.16719666666666663, -0.32093666666666676, 0...","[-0.1671633243560791, -0.4068999926249186, 0.4...","[0.08763999999999997, -0.40253666666666654, -0...","[0.000200004572619993, 0.08608333614031476, 0....",10.500000


In [69]:
fig = px.histogram(df_results_mean_histo, 
                   x = "lambda_str",
                   y = "diff_count",
                   color="algorithm",
                   barmode="group",
                   title=f"{params['data']}_{params['base_model']}_alpha=0.1_histogram")
fig.show()

In [70]:
'#636EFA',
'#EF553B',
'#00CC96',
'#AB63FA',
"#8EF1F3",
"#F6B08C",
"#B5FFBE",
"#D7BBF4"

custom_colors = {"Alg1" : '#636EFA', "L1PSD" : '#EF553B',"ROARLInf" : '#00CC96', "ROARL1" : '#AB63FA'}
algorithm_latex_map = {"Alg1" : r"$\text{Alg}2\ (L^\infty)\ (\alpha=0.1)$",
                "L1PSD" : r"$\text{Alg}1\ (L^1)\ (\alpha=0.1)$",
                "ROARLInf" : r"$\text{ROAR}\ (L^\infty)\ (\alpha=0.1)$",
                "ROARL1" : r"$\text{ROAR}\ (L^1)\ (\alpha=0.1)$"}
params["algorithms"] = ["L1PSD", "Alg1", "ROARL1", "ROARLInf"]

In [72]:
fig = go.Figure()
font_family = 'Times New Roman'
font_color = 'black'
font_size = 20
# width, height = 720, 540
width, height = 720, 540

for algo in params['algorithms']:
    df_sub = df_results_mean_histo[df_results_mean_histo["algorithm"] == algo]
    fig.add_trace(
        go.Bar(
            x=df_sub["lambda_str"],
            y=df_sub["diff_count"],
            name=algorithm_latex_map[algo],
            marker=dict(color=custom_colors[algo])
        )
    )

fig.update_layout(plot_bgcolor="white",
                  paper_bgcolor="white",)
                #   xaxis=dict(title="Lambda",
                #              showgrid=True,
                #              mirror=True,
                #              gridcolor="black",),
                #     yaxis=dict(title="sum of diff_count",
                #                showgrid=True,
                #                gridcolor="black",
                #                mirror=True))
fig.update_xaxes(
        title=dict(
            text='Lambda',
            font=dict(
                family=font_family,
                color=font_color,
                size=font_size
            )
            ), 
        showline=True, 
        mirror=True,
        showgrid=False,
        linecolor='black', 
        gridcolor='lightgrey', 
        zerolinewidth=1,
        zerolinecolor='lightgrey',
        )

fig.update_yaxes(
        title=dict(
            text='No. of Features Changed',
            font=dict(
                family=font_family,
                color=font_color,
                size=font_size
            ), 
            ), 
        showline=True, 
        mirror=True,
        linecolor='black', 
        gridcolor='lightgrey',
        zerolinewidth=1,
        zerolinecolor='lightgrey',
        )

fig.update_layout(
        width=width,
        height=height,
        plot_bgcolor='white',
        paper_bgcolor='white',
        margin=dict(t=50,b=25,l=25,r=25),
        # title =dict(
        #     text=f"{params['data'].capitalize()} | {params['base_model'].upper()} | WC ({params['adv_method']}) Adversary", 
        #     x= 0.5, 
        #     font=dict(family=font_family, size=20)
        #     ),
        legend=dict(
            # x=0.975,
            x=0.5, 
            y=0.975, 
            orientation='v',
            xanchor='right',
            font=dict(
                family=font_family,
                color=font_color,
                size=15
                ), 
            bgcolor='rgba(255, 255, 255, 0.7)',
            bordercolor='lightgrey',
            borderwidth=1,
            entrywidth=100.5,
            ),
        xaxis=dict(
            tickfont=dict(
                family=font_family,
                color=font_color,
                size=20,
            ),
            # range=[0,25]
            
        ),
        yaxis=dict(
            tickfont=dict(
                family=font_family,
                color=font_color,
                size=20
            ),
            range=[0,25]
        )
    )

In [ ]:
# figName = f"sparsity-model_{params['base_model']}-dataset_{params['data']}-alpha_0.1" 
# fig.write_image(f"C:\\Users\\pmyat\\OneDrive - Drexel University\\ResearchAssistantCoop_Jabbari\\" +
# f"experimentResults\\figures\\" + 
# figName + f".pdf")

In [285]:
from plotly.subplots import make_subplots

width_oneF = 150
height_oneF = 1000 / len(params["algorithms"])

fig_sub = make_subplots(rows = len(sparsity_lambda), 
                        cols = 1, 
                        shared_xaxes=True,
                        subplot_titles=[f"Lambda {val}" for val in sparsity_lambda],
                        x_title="Features",
                        y_title="Algorithms")

for i,lamb in enumerate(sparsity_lambda):
    df_results_mean_tmp = df_results_mean[df_results_mean['lambda'] == lamb]
    if "ROARLInf" in params['algorithms'] and "ROARL1" in params['algorithms']:
        df_results_mean_tmp.iloc[[0,1]] = df_results_mean_tmp.iloc[[1,0]]
    stacked  = np.stack(df_results_mean_tmp['diff'])

    fig_sub.add_trace(go.Heatmap(z=stacked.round(2),
                                x=np.arange(stacked.shape[1]),
                                y=df_results_mean_tmp['algorithm'].to_list(),
                                coloraxis="coloraxis",
                                texttemplate="%{z}"), 
                                row=i+1, col=1)

fig_sub.update_xaxes(tickmode="array", 
                     tickvals=np.arange(stacked.shape[1]), 
                     row=len(sparsity_lambda), 
                     col=1)
# fig_sub.update_yaxes(title_text="Algorithms", row=len(sparsity_lambda) // 2, col=1)
fig_sub.update_layout(
    coloraxis=dict(colorscale="Reds"),
    coloraxis_colorbar=dict(
        title="Avg. Cost",
    ),
    width=width_oneF * stacked.shape[1],
    height=height_oneF * stacked.shape[0],
    title_text = f"{params['data']}_{params['base_model']}_alpha=0.1_Sparsity"
)

In [ ]:
# figNameHisto = f"{params['base_model']}_{params['data']}_histogram.html" 
# figNameSparsity = f"{params['base_model']}_{params['data']}_sparsity.html"
# fig.write_html(f"C:\\Users\\pmyat\\OneDrive - Drexel University\\ResearchAssistantCoop_Jabbari\\" +
#       f"experimentResults\\meetings\\2025-09-11\\" + 
#       figNameHisto + f".html")
# fig_sub.write_html(f"C:\\Users\\pmyat\\OneDrive - Drexel University\\ResearchAssistantCoop_Jabbari\\" +
#       f"experimentResults\\meetings\\2025-09-11\\" + 
#       figNameSparsity + f".html")

In [278]:
df_results[(df_results['seed'] == 0) & (df_results['i'] == 24) & (df_results['lambda'] == 0.001)]

,algorithm,seed,alpha,lambda,i,x_0,x_r,theta_0,diff,diff_count
175,Alg1,0,0.1,0.001,24,"[1.2526, -0.3361, -1.0155, 0.0, 0.0, 0.0, 1.0]","[1.2526, 5.1217, -1.0155, 0.0, 0.0, 0.0, 1.0]","[-1.3217, 1.6803, 0.7039, -0.1972, -0.0802, 0....","[0.0, 5.4578, 0.0, 0.0, 0.0, 0.0, 0.0]",1
385,L1PSD,0,0.1,0.001,24,"[1.2526, -0.3361, -1.0155, 0.0, 0.0, 0.0, 1.0]","[-0.728, 2.7053, -1.0155, 0.0, 0.0, -0.0, 1.0]","[-1.4307, 1.8215, 0.7708, -0.1706, -0.1865, 0....","[1.9806, 3.0414, 0.0, 0.0, 0.0, 0.0, 0.0]",2
